# 04 — ECSS Aerospace Compliance (Souffle)

Demonstrates Souffle adapter with ECSS ESSB-ST-U-007 space debris mitigation:
- ECSS domain schema with typed predicates
- Compliance rules with `condition_weights` + `RuleRef` chains
- `sdk.evaluate()` → `sdk.accept()` → `sdk.export_package()`
- Certainty summary and narrative via `AuditQuery`
- Souffle provenance: proof trees via adapter API
- Audit package + static site generation

**Prerequisites:** [01](01_sdk_basics.ipynb)–[03](03_certainty_and_evidence_tree.ipynb).  
**Next:** [05_dora_pyreason_propagation.ipynb](05_dora_pyreason_propagation.ipynb)

## 1. Schema: Mission + ECSS Predicates

In [ ]:
from __future__ import annotations
import sys, tempfile
from pathlib import Path
from pprint import pprint

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from factpy_kernel.sdk import (
    SDKStore, SDKRegistry, Entity, Identity, Field,
    Rule, Derivation, Pred, vars as sdk_vars,
)
from factpy_kernel.sdk.dsl.rule import RuleRef
from factpy_kernel.domains.ecss import (
    ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
    ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID,
    ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
    ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID,
)
from factpy_kernel.adapters.souffle.package import ExportOptions
from factpy_kernel.audit import AuditQuery, load_audit_package

In [ ]:
class Mission(Entity):
    mission_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    orbit_type: str = Field(cardinality="single")
    mission_profile: str = Field(cardinality="single")
    passivation_status: str = Field(cardinality="single")
    overall_compliance_status: str = Field(cardinality="single")

class Ecss(Entity):
    anchor_id: str = Identity(primary_key=True)
    collision_probability_ppm: int = Field(cardinality="single")
    collision_probability_threshold_ppm: int = Field(cardinality="single")
    disposal_success_probability_ppm: int = Field(cardinality="single")
    disposal_success_threshold_ppm: int = Field(cardinality="single")
    requirement: str = Field(cardinality="single")
    verification_method: str = Field(cardinality="multi")
    compliance_status: str = Field(cardinality="single")

sdk = SDKStore([Mission, Ecss])
ecss_preds = [p["pred_id"] for p in sdk.schema_ir["predicates"] if p["pred_id"].startswith("ecss:")]
print(f"Schema: {ecss_preds}")

## 2. Rules: ESSB-ST-U-007 Compliance Checks

In [ ]:
with sdk_vars("m", "prob", "threshold") as (m, prob, threshold):
    disposal_check = Rule(id="q.essb_u007_disposal_check", version="1.0.0",
        select=[m, prob],
        where=[Pred(ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID, m, prob),
               Pred(ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID, m, threshold),
               prob >= threshold],
        expose=True, condition_weights={"b0.a0": 0.8, "b0.a1": 0.5})

with sdk_vars("m", "prob", "threshold") as (m, prob, threshold):
    collision_check = Rule(id="q.essb_u007_collision_check", version="1.0.0",
        select=[m, prob],
        where=[Pred(ECSS_COLLISION_PROBABILITY_PPM_PRED_ID, m, prob),
               Pred(ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID, m, threshold),
               threshold >= prob],
        expose=True, condition_weights={"b0.a0": 0.9, "b0.a1": 0.4})

with sdk_vars("m", "status") as (m, status):
    passivation_check = Rule(id="q.essb_u007_passivation_check", version="1.0.0",
        select=[m, status],
        where=[Pred("mission:passivation_status", m, status), status == "complete"],
        expose=True, condition_weights={"b0.a0": 1.0})

with sdk_vars("m", "profile", "status", "prob", "cpb", "ps") as (m, profile, status, prob, cpb, ps):
    compliance_rule = Rule(id="q.essb_u007_overall_compliance", version="1.0.0",
        select=[m, status],
        where=[Pred("mission:mission_profile", m, profile), profile == "single",
               RuleRef(disposal_check)(m, prob),
               RuleRef(collision_check)(m, cpb),
               RuleRef(passivation_check)(m, ps),
               status == "compliant"],
        expose=True)

all_rules = [disposal_check, collision_check, passivation_check, compliance_rule]
print(f"Defined {len(all_rules)} rules")

## 3. Registry & Mission Data

In [ ]:
registry_dir = tempfile.mkdtemp(prefix="ecss_demo_")
registry = SDKRegistry(root_dir=Path(registry_dir))
registry.apply_schema_classes([Mission, Ecss])
for rule in all_rules:
    registry.register_rule(rule, schema_ir=sdk.schema_ir)

sentinel_ref = sdk.ref(Mission, mission_id="SENTINEL-7", locale="en")

with sdk.batch(meta={"source": "ecss_demo"}) as tx:
    sentinel = tx.entity(Mission, mission_id="SENTINEL-7", locale="en")
    sentinel.name.set("Sentinel-7 LEO Observatory")
    sentinel.orbit_type.set("LEO")
    sentinel.mission_profile.set("single")
    sentinel.passivation_status.set("complete", meta={"confidence": 0.99})
    tx.commit()

# Write ECSS measurement facts via low-level SDK set
sdk.set(Ecss.disposal_success_probability_ppm, sentinel_ref, 920000,
        meta={"confidence": 0.85})
sdk.set(Ecss.disposal_success_threshold_ppm, sentinel_ref, 900000)
sdk.set(Ecss.collision_probability_ppm, sentinel_ref, 500,
        meta={"confidence": 0.70})
sdk.set(Ecss.collision_probability_threshold_ppm, sentinel_ref, 1000)

print(f"SENTINEL-7: single/LEO, disposal=920k>=900k, collision=500<=1000")

## 4. Evaluate: Sub-Checks + Overall Compliance

In [ ]:
def evaluate_check(label, rule, target, extra_where=None):
    with sdk_vars("m", "val") as (m, val):
        where_clauses = [RuleRef(rule)(m, val)]
        if extra_where: where_clauses.extend(extra_where)
        drv = Derivation(id=f"drv.{label}", version="1.0.0",
            where=where_clauses, target=target, head_vars=[m, val])
    cands = sdk.evaluate(drv, mode="native", registry=registry)
    if cands:
        c = cands[0]
        print(f"  {label}: PASSED (confidence_kind={c.confidence_kind})")
        return cands
    else:
        print(f"  {label}: FAILED")
        return []

cands_disposal = evaluate_check("disposal",
    disposal_check, ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID)
cands_collision = evaluate_check("collision",
    collision_check, ECSS_COLLISION_PROBABILITY_PPM_PRED_ID)
cands_passivation = evaluate_check("passivation",
    passivation_check, "mission:passivation_status")

# Overall compliance
with sdk_vars("m", "status") as (m, status):
    overall_drv = Derivation(id="drv.overall", version="1.0.0",
        where=[RuleRef(compliance_rule)(m, status)],
        target="mission:overall_compliance_status", head_vars=[m, status])
cands_overall = sdk.evaluate(overall_drv, mode="native", registry=registry)
if cands_overall:
    print(f"  Overall: PASSED (confidence_kind={cands_overall[0].confidence_kind})")

## 5. Accept + Export Audit Package

In [ ]:
# Accept all candidates
for cands in [cands_disposal, cands_collision, cands_passivation, cands_overall]:
    for c in cands:
        sdk.accept(c, approved_by="ecss_demo")

# Export audit package
audit_dir = tempfile.mkdtemp(prefix="ecss_audit_")
sdk.export_package(audit_dir, ExportOptions(package_kind="audit"), registry=registry)

pkg = load_audit_package(audit_dir)
aq = AuditQuery(pkg)
candidates = aq.list_candidates()
print(f"Audit: {len(candidates)} candidates")

for row in candidates[:4]:
    cid = row["candidate_id"]
    cs = aq.get_candidate_certainty_summary(cid)
    if cs:
        print(f"  {cid[:30]}... certainty={cs['aggregate_certainty']}")

## 6. Souffle Provenance: Proof Trees

Use adapter-level Souffle API for proof tree extraction.

In [ ]:
from factpy_kernel.adapters.souffle.provenance import run_package_provenance
from factpy_kernel.adapters.souffle.runner import run_package

# Export inference package for Souffle
prov_dir = tempfile.mkdtemp(prefix="ecss_prov_")
sdk.export_package(prov_dir, ExportOptions(package_kind="inference"), registry=registry)

try:
    run_package(Path(prov_dir), ["__query__"], engine="souffle")
    query_rel = "q_essb_u007_disposal_check"
    out_path = Path(prov_dir) / "outputs" / f"{query_rel}.out.facts"
    if out_path.exists():
        rows = [line.split("\t") for line in out_path.read_text().splitlines() if line.strip()]
        if rows:
            prov_query = query_rel + "(" + ", ".join(f'\"{c}\"' for c in rows[0]) + ")"
            trees = run_package_provenance(Path(prov_dir), [prov_query])
            if trees:
                tree = trees[0]
                print(f"=== Souffle Proof Tree: {tree.query} ===")
                def print_proof(node, indent=0):
                    args = ", ".join(node.args[:3])
                    label = f"{node.relation}({args})"
                    print(f"{'  '*indent}{'FACT' if node.node_type == 'axiom' else f'RULE {node.rule_number}'}: {label}")
                    for child in node.children:
                        print_proof(child, indent+1)
                print_proof(tree.root)
except FileNotFoundError:
    print("Souffle CLI not found. Install souffle for proof tree extraction.")

## 7. Static Audit Site

In [ ]:
from factpy_kernel.audit.static_ui import render_audit_static_site

site_dir = tempfile.mkdtemp(prefix="ecss_site_")
render_audit_static_site(audit_dir, site_dir)
pages = sorted(Path(site_dir).rglob("*.html"))
print(f"Static site: {len(pages)} pages at {site_dir}")
for p in pages[:10]:
    print(f"  {p.relative_to(site_dir)}")

---
**Next:** [05_dora_pyreason_propagation.ipynb](05_dora_pyreason_propagation.ipynb)